# ผู้ส่ง
นายรัชพล สม่าหลี
รหัส 600659

ต้นฉบับ:

https://colab.research.google.com/drive/1z7VZUyvTLQo5DtpE2kzc8swanSihnu_F?usp=sharing



---

In [ ]:
import sys
!{sys.executable} -m pip install -q --root-user-action=ignore \
    numpy \
    pandas \
    torch \
    scipy \
    tqdm \
    librosa \
    scikit-learn \
    matplotlib \
    joblib


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip


# Methodology

ปรับจาก https://openreview.net/forum?id=ebRFWvAUjS


## 1. การเลือกสัญญาณและปรับข้อมูล (Signal Selection & Normalization)
- **สัญญาณหลัก:** ACC (X,Y,Z), BVP, EDA, TEMP (ตัด HR, IBI ทิ้ง)
- **EDA (Phasic):** สกัดเฉพาะส่วน Phasic
- **TEMP (Delta):** แปลงข้อมูลด้วยต่างลำดับที่หนึ่ง (First-order difference)
- **Normalization:**
  - Median centering + Adaptive IQR (Sliding window 300s)
  - Outlier clipping: ตัดที่ 20x IQR (ACC, BVP, EDA) และ 15x IQR (TEMP)
  - Z-score normalization

## 2. การปรับความถี่และการแปลง Spectrogram
- **Resampling:** จัดการข้อมูลใหม่จาก 16Hz เป็น **32Hz**
- **Feature Extraction:** ใช้ **STFT (Short-Time Fourier Transform)** แปลงเป็น Spectrogram

## 3. การจัดโครงสร้างข้อมูล (Input Structuring)
- **Epoching:** 30 วินาที/Epoch
- **Segment Size:** รวม 1,024 Epochs (8.5 ชม.) รวดเดียวต่อ 1 ตัวอย่าง (Seq-to-Seq)
- **Padding:** Zero-padding หากข้อมูลสั้นกว่า 1,024 Epochs

## 4. โครงสร้างโมเดล (2D U-Net Architecture)
- **M = 10:** จำนวนชุด Encoder / Decoder
- **K = 16:** Kernel height สำหรับ 2D Convolutions
- **Initial Filters = 32:** เริ่มต้นที่ 32 ตัวกรอง ตั้ง MAX ที่ 256

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from itertools import product as iterproduct
import os
import glob
from tqdm import tqdm
from scipy import signal
import librosa
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score
import gc
from joblib import Parallel, delayed

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'  GPU : {torch.cuda.get_device_name(0)}')
    print(f'  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('  Running on CPU')

Device: cuda
  GPU : NVIDIA GeForce RTX 4090
  VRAM: 25.3 GB


## 1. Search Space (Hyperparameters)

In [ ]:
SEARCH_SPACE = {
    # กำหนดรูปแบบข้อมูลที่ใช้
    'epochs_per_night': [1024],   # ใช้ข้อมูล 1 คืน ประมาณ 1024 ช่วงเวลา
    'n_classes':        [5],      # จำนวนคลาสที่ต้องทำนาย (เช่น sleep stage)

    # ตั้งค่าโครงสร้างโมเดล
    'M':               [10],      # ความลึกของโมเดล (จำนวนชั้น)
    'K':               [16],      # ขนาด kernel ที่ใช้ใน convolution
    'initial_filters': [32],      # จำนวน filter เริ่มต้น
    'max_filters':     [256],     # จำกัดจำนวน filter สูงสุด ไม่ให้ใหญ่เกินไป

    # ตั้งค่าการ train โมเดล
    'learning_rate':   [1e-3],    # ความเร็วในการเรียนรู้
    'focal_alpha':     [0.15],    # ค่าถ่วงน้ำหนักของ focal loss
    'focal_gamma':     [2.0],     # เน้นตัวอย่างที่ยากมากขึ้น
    'max_epochs':      [50],      # train กี่รอบ
    'batch_size':      [1],       # ใช้ทีละตัว เพราะข้อมูลใหญ่มาก (กิน memory สูง)
}

# เอาชื่อ parameter ออกมาเป็น list
KEYS = list(SEARCH_SPACE.keys())

# สร้างทุก combination ของค่าที่เป็นไปได้
combos = list(iterproduct(*[SEARCH_SPACE[k] for k in KEYS]))

# ดูว่ามีทั้งหมดกี่แบบ
print(f'Total combinations: {len(combos)}')

Total combinations: 1


## 2. Model Definition
M=10 Encoder/Decoder blocks, Kernel Height (Time) = 16, Initial Filters = 32

In [ ]:
class ConvBlock(nn.Module):
    # บล็อกพื้นฐานของโมเดล
    # รับข้อมูลเข้าไป แล้วแปลงให้อยู่ในรูปที่โมเดลใช้งานต่อได้
    def __init__(self, in_ch, out_ch, kernel_height=16):
        super().__init__()
        self.conv = nn.Sequential(
            # ดึงลักษณะสำคัญจากข้อมูล
            nn.Conv2d(in_ch, out_ch, kernel_size=(kernel_height, 3), padding='same'),

            # ปรับค่าให้การ train เสถียรมากขึ้น
            nn.BatchNorm2d(out_ch),

            # ใส่ความไม่เป็นเส้นตรงให้โมเดลเรียนรู้ pattern ได้ดีขึ้น
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)


class GnaraUNet(nn.Module):
    # โมเดลหลัก
    # โครงสร้างนี้จะค่อยๆ ย่อข้อมูลเพื่อดึง pattern
    # แล้วค่อยขยายกลับมาเพื่อทำนายผลในแต่ละช่วงเวลา
    def __init__(self, cfg, in_channels):
        super().__init__()

        self.M = cfg['M']
        self.K = cfg['K']
        init_f = cfg['initial_filters']
        max_f = cfg.get('max_filters', 256)  # จำกัดไม่ให้จำนวน filter โตเกินไป
        self.n_classes = cfg['n_classes']

        # คำนวณจำนวน filter ของแต่ละชั้น
        # ตอนลึกลงจะเพิ่มขึ้นเรื่อยๆ แต่ไม่เกิน max_f
        filters = []
        current_f = init_f
        for i in range(self.M + 1):
            filters.append(min(current_f, max_f))
            current_f *= 2

        # ส่วนขาลง
        # ใช้ดึง feature สำคัญจากข้อมูล และค่อยๆ ลดขนาดลง
        self.encoders = nn.ModuleList()
        self.pools = nn.ModuleList()

        in_ch = in_channels
        for i in range(self.M):
            self.encoders.append(ConvBlock(in_ch, filters[i], self.K))
            self.pools.append(nn.MaxPool2d(kernel_size=(2, 1)))
            in_ch = filters[i]

        # ส่วนล่างสุดของโมเดล
        # เป็นจุดที่ข้อมูลถูกย่อจนได้ feature ที่เข้มข้นที่สุด
        self.bottom = ConvBlock(filters[-2], filters[-1], self.K)

        # ส่วนขาขึ้น
        # ค่อยๆ ขยายข้อมูลกลับขึ้นมา เพื่อให้ทำนายผลได้ละเอียดอีกครั้ง
        self.upconvs = nn.ModuleList()
        self.decoders = nn.ModuleList()
        for i in reversed(range(self.M)):
            self.upconvs.append(
                nn.ConvTranspose2d(filters[i+1], filters[i], kernel_size=(2, 1), stride=(2, 1))
            )

            # ตอนขยายกลับ จะเอาข้อมูลจากขาลงมาช่วยด้วย
            self.decoders.append(ConvBlock(filters[i] * 2, filters[i], self.K))

        # ส่วนสุดท้ายสำหรับแปลง feature ให้เป็นผลทำนายของแต่ละคลาส
        self.classifier = nn.Sequential(
            nn.Conv2d(init_f, init_f, kernel_size=1),
            nn.ReLU(inplace=True),

            # ปรับความยาวผลลัพธ์ให้ตรงกับจำนวนช่วงเวลาที่ต้องการ
            nn.AdaptiveAvgPool2d((1024, 1)),

            # แปลงเป็นจำนวนคลาสที่ต้องการทำนาย
            nn.Conv2d(init_f, self.n_classes, kernel_size=1)
        )

    def forward(self, x):
        # เก็บข้อมูลจากขาลงไว้ เพื่อเอาไปช่วยตอนขาขึ้น
        skips = []

        # ขาลง: ดึง feature แล้วลดขนาดข้อมูล
        for i in range(self.M):
            x = self.encoders[i](x)
            skips.append(x)
            x = self.pools[i](x)

        # ผ่านจุดล่างสุดของโมเดล
        x = self.bottom(x)

        # ขาขึ้น: ขยายข้อมูลกลับ และรวมกับข้อมูลเดิมจากขาลง
        for i, (up, dec) in enumerate(zip(self.upconvs, self.decoders)):
            x = up(x)
            skip = skips[-(i+1)]

            # ถ้าขนาดไม่ตรงกัน ปรับให้ตรงก่อน
            if x.shape[2] != skip.shape[2]:
                x = F.interpolate(x, size=(skip.shape[2], skip.shape[3]))

            # รวมข้อมูลจากขาขึ้นกับขาลง
            x = torch.cat([x, skip], dim=1)

            # ผ่าน block เพื่อรวมข้อมูลให้พร้อมใช้ต่อ
            x = dec(x)

        # แปลงเป็นผลทำนายสุดท้าย
        x = self.classifier(x)

        # ตัดมิติที่ไม่จำเป็นออก
        x = x.squeeze(-1)

        return x

## 3. Focal Loss & Training Utilities
เปเปอร์ใช้ Binary Focal Loss เพื่อแก้ปัญหา Class Imbalance

In [ ]:
class FocalLoss(nn.Module):
    # loss function แบบพิเศษ
    # เอาไว้ช่วยแก้ปัญหา class imbalance (บางคลาสมีน้อย)
    def __init__(self, alpha=0.15, gamma=2.0, reduction='mean', ignore_index=-100):
        super().__init__()
        self.alpha = alpha      # น้ำหนักของ loss
        self.gamma = gamma      # เน้นตัวอย่างที่ยาก
        self.reduction = reduction
        self.ignore_index = ignore_index  # ใช้สำหรับ ignore padding

    def forward(self, inputs, targets):
        # inputs = ค่าที่โมเดลทำนาย (logits)
        # targets = คำตอบจริง

        # คำนวณ cross entropy ปกติก่อน
        ce_loss = F.cross_entropy(
            inputs, targets,
            reduction='none',
            ignore_index=self.ignore_index
        )

        # คำนวณว่าโมเดลมั่นใจแค่ไหน (pt สูง = ทายง่าย)
        pt = torch.exp(-ce_loss)

        # ปรับ loss ให้โฟกัสกับตัวอย่างที่ยากมากขึ้น
        focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss

        if self.reduction == 'mean':
            # เอาค่าเฉลี่ยเฉพาะตำแหน่งที่ไม่ใช่ padding
            mask = (targets != self.ignore_index)
            return focal_loss[mask].mean() if mask.any() else focal_loss.sum()

        elif self.reduction == 'sum':
            return focal_loss.sum()

        return focal_loss


def train_epoch(model, loader, optimizer, criterion, device, scaler):
    # ฟังก์ชัน train โมเดล 1 รอบ

    model.train()
    total_loss, n_batches = 0.0, 0

    for Xb, yb in loader:
        # ย้ายข้อมูลไป GPU
        Xb = Xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)

        # เคลียร์ gradient เก่า
        optimizer.zero_grad(set_to_none=True)

        # ใช้ mixed precision เพื่อให้เร็วขึ้นและกิน memory น้อยลง
        with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
            logits = model(Xb)        # โมเดลทำนาย
            loss = criterion(logits, yb)  # คำนวณ loss

        # ย้อนกลับเพื่อปรับค่าโมเดล
        scaler.scale(loss).backward()

        # อัปเดตน้ำหนักโมเดล
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        n_batches += 1

    # คืนค่า loss เฉลี่ย
    return total_loss / n_batches


def evaluate_f1(model, loader, device, ignore_index=-100):
    # ฟังก์ชันสำหรับวัดผลโมเดลด้วย F1-score

    model.eval()
    preds, labels = [], []

    with torch.no_grad():
        for Xb, yb in loader:
            # ย้าย input ไป GPU
            Xb = Xb.to(device, non_blocking=True)

            # ใช้ mixed precision เหมือนตอน train
            with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
                logits = model(Xb)

            # เลือกคลาสที่โมเดลทำนายได้ดีที่สุด
            pred = logits.argmax(1).cpu().numpy()
            true = yb.numpy()

            # เอาเฉพาะตำแหน่งที่ไม่ใช่ padding
            mask = (true != ignore_index)

            preds.extend(pred[mask])
            labels.extend(true[mask])

    # คำนวณ F1-score แบบรวมทุกคลาส
    return f1_score(labels, preds, average='weighted', zero_division=0)

# โค้ดสำหรับการ train หลังจาหา Hyper parameter

## 4. Data Preprocessing

In [ ]:
# --- Configuration ---
TRAIN_DIR = r'./data/train/train'
CACHE_DIR = r'./preprocessed_cache'
os.makedirs(CACHE_DIR, exist_ok=True)

FS_ORIGINAL = 16
FS_TARGET   = 32
EPOCH_SEC   = 30
EPOCH_SIZE  = FS_TARGET * EPOCH_SEC
MAX_EPOCHS  = 1024
IGNORE_IDX  = -100

# map label จากตัวอักษรเป็นตัวเลข
LABEL_MAP = {'W':0, 'N1':1, 'N2':2, 'N3':3, 'R':4}

def extract_phasic_eda(eda_signal, fs):
    # แยกส่วน phasic ของ EDA
    sos = signal.butter(2, 0.05, 'highpass', fs=fs, output='sos')
    return signal.sosfiltfilt(sos, eda_signal)

def adaptive_iqr_normalize(sig, fs, window_sec=300, clip_multiplier=20):
    # normalize แบบดูค่าจากช่วงรอบข้างของสัญญาณ
    w_size = int(window_sec * fs)
    s_series = pd.Series(sig)

    median = s_series.rolling(window=w_size, center=True, min_periods=1).median().values
    q75 = s_series.rolling(window=w_size, center=True, min_periods=1).quantile(0.75).values
    q25 = s_series.rolling(window=w_size, center=True, min_periods=1).quantile(0.25).values

    iqr = q75 - q25
    iqr[iqr == 0] = 1e-6

    # ตัดค่าที่โดดเกินไปออก
    upper_bound = median + (clip_multiplier * iqr)
    lower_bound = median - (clip_multiplier * iqr)
    sig_clipped = np.clip(np.asarray(sig), lower_bound, upper_bound)

    # ปรับให้อยู่ในสเกลเดียวกัน
    mean, std = np.mean(sig_clipped), np.std(sig_clipped)
    if std == 0:
        std = 1

    return np.array((sig_clipped - mean) / std, dtype=np.float32)

def generate_spectrogram(sig, fs):
    # แปลงสัญญาณเวลาเป็น spectrogram
    sig = np.asarray(sig, dtype=np.float32)
    stft = librosa.stft(sig, n_fft=256, hop_length=64)
    spec = librosa.power_to_db(np.abs(stft)**2, ref=np.max)
    return spec.astype(np.float32)

def process_one_night(file_path):
    # ประมวลผลข้อมูล 1 คืน

    subject_id = os.path.splitext(os.path.basename(file_path))[0]
    df_raw = pd.read_csv(file_path)

    # เตรียม TEMP โดยใช้ค่าการเปลี่ยนแปลงแทนค่าเดิม
    temp = df_raw['TEMP'].values.astype(np.float32)
    temp_diff = np.diff(temp, prepend=temp[0]).astype(np.float32)

    # เพิ่ม sampling rate จาก 16 เป็น 32
    data = {
        'acc_x': np.repeat(df_raw['ACC_X'].values.astype(np.float32), 2),
        'acc_y': np.repeat(df_raw['ACC_Y'].values.astype(np.float32), 2),
        'acc_z': np.repeat(df_raw['ACC_Z'].values.astype(np.float32), 2),
        'bvp': np.repeat(df_raw['BVP'].values.astype(np.float32), 2),
        'eda': np.repeat(df_raw['EDA'].values.astype(np.float32), 2),
        'temp_diff': np.repeat(temp_diff, 2)
    }
    labels_raw = np.repeat(df_raw['Sleep_Stage'].values, 2)

    # เตรียม EDA และ normalize ทุกสัญญาณ
    data['eda_phasic'] = extract_phasic_eda(data['eda'], FS_TARGET)

    for k in ['acc_x', 'acc_y', 'acc_z', 'bvp', 'eda_phasic']:
        data[k] = adaptive_iqr_normalize(data[k], FS_TARGET, clip_multiplier=20)

    data['temp_diff'] = adaptive_iqr_normalize(data['temp_diff'], FS_TARGET, clip_multiplier=15)

    # แปลงทุกสัญญาณเป็น spectrogram
    specs = [
        generate_spectrogram(data[k], FS_TARGET)
        for k in ['acc_x', 'acc_y', 'acc_z', 'bvp', 'eda_phasic', 'temp_diff']
    ]
    night_specs = np.stack(specs)

    # แบ่งเป็น epoch
    frames_per_epoch = 15
    total_epochs = night_specs.shape[-1] // frames_per_epoch
    night_specs = night_specs[:, :, :total_epochs * frames_per_epoch]

    # reshape ให้อยู่ในรูปที่โมเดลใช้
    night_epochs = np.zeros((90, total_epochs, 129), dtype=np.float16)
    for e in range(total_epochs):
        epoch_slice = night_specs[:, :, e * 15 : (e + 1) * 15]
        night_epochs[:, e, :] = epoch_slice.transpose(0, 2, 1).reshape(90, 129).astype(np.float16)

    # สร้าง label ของแต่ละ epoch โดยดูคลาสที่เจอบ่อยที่สุด
    epoch_labels = []
    for e in range(total_epochs):
        lbl_slice = labels_raw[e * EPOCH_SIZE : (e + 1) * EPOCH_SIZE]
        u, c = np.unique(lbl_slice, return_counts=True)
        epoch_labels.append(LABEL_MAP.get(u[np.argmax(c)], 0))
    epoch_labels = np.array(epoch_labels, dtype=np.int8)

    # padding ให้ทุกคืนยาวเท่ากัน
    if total_epochs < MAX_EPOCHS:
        pad_len = MAX_EPOCHS - total_epochs
        night_epochs = np.pad(night_epochs, ((0, 0), (0, pad_len), (0, 0)), mode='constant')
        epoch_labels = np.pad(epoch_labels, (0, pad_len), mode='constant', constant_values=IGNORE_IDX)
    else:
        night_epochs = night_epochs[:, :MAX_EPOCHS, :]
        epoch_labels = epoch_labels[:MAX_EPOCHS]

    return night_epochs, epoch_labels, subject_id

# --- Main Execution ---
train_files = sorted(glob.glob(os.path.join(TRAIN_DIR, '*.csv')))
cache_x = os.path.join(CACHE_DIR, 'X_all.npy')
cache_y = os.path.join(CACHE_DIR, 'y_all.npy')
cache_g = os.path.join(CACHE_DIR, 'groups.npy')

if os.path.exists(cache_x) and os.path.exists(cache_y) and os.path.exists(cache_g):
    print("⚡ Loading preprocessed data from cache...")
    X_all = np.load(cache_x)
    y_all = np.load(cache_y)
    groups = np.load(cache_g)
else:
    print(f"🚀 Processing {len(train_files)} files in parallel...")
    results = Parallel(n_jobs=-1)(delayed(process_one_night)(f) for f in tqdm(train_files))

    X_all  = np.array([r[0] for r in results])
    y_all  = np.array([r[1] for r in results])
    groups = np.array([r[2] for r in results])

    print("💾 Saving preprocessed data to cache...")
    np.save(cache_x, X_all)
    np.save(cache_y, y_all)
    np.save(cache_g, groups)

print(f'\nDone! Data Shape: {X_all.shape}, Label Shape: {y_all.shape}')

⚡ Loading preprocessed data from cache...

Done! Data Shape: (83, 90, 1024, 129), Label Shape: (83, 1024)


## 5. Training Loop (GroupKFold Seq-to-Seq)

In [ ]:
gkf = GroupKFold(n_splits=5)
PATIENCE = 5
FIXED_KEYS = {'epochs_per_night', 'n_classes', 'max_epochs', 'batch_size'}

all_results = []
IN_CHANNELS = 90

for combo_idx, combo in enumerate(combos):
    cfg = dict(zip(KEYS, combo))

    print(f"\n{'='*65}\n  Combo [{combo_idx+1}/{len(combos)}]")
    for k, v in cfg.items():
        if k not in FIXED_KEYS:
            print(f'    {k}: {v}')

    fold_f1s = []

    for fold, (train_idx, val_idx) in enumerate(gkf.split(X_all, y_all, groups=groups)):
        # เคลียร์ memory ก่อนเริ่ม fold ใหม่
        gc.collect()
        torch.cuda.empty_cache()

        X_train_f, X_val_f = X_all[train_idx], X_all[val_idx]
        y_train_f, y_val_f = y_all[train_idx], y_all[val_idx]

        pin = device.type == 'cuda'

        # แปลงข้อมูลเป็น tensor ตอนส่งเข้า DataLoader
        train_dl = DataLoader(
            TensorDataset(
                torch.tensor(X_train_f, dtype=torch.float32),
                torch.tensor(y_train_f, dtype=torch.long)
            ),
            batch_size=cfg['batch_size'],
            shuffle=True,
            pin_memory=pin,
            drop_last=True
        )

        val_dl = DataLoader(
            TensorDataset(
                torch.tensor(X_val_f, dtype=torch.float32),
                torch.tensor(y_val_f, dtype=torch.long)
            ),
            batch_size=cfg['batch_size'],
            shuffle=False,
            pin_memory=pin
        )

        # สร้างโมเดลและส่วนที่ใช้ train
        model = GnaraUNet(cfg, in_channels=IN_CHANNELS).to(device)
        optimizer = optim.Adam(model.parameters(), lr=cfg['learning_rate'])
        criterion = FocalLoss(
            alpha=cfg['focal_alpha'],
            gamma=cfg['focal_gamma'],
            ignore_index=IGNORE_IDX
        )

        # ใช้ mixed precision scaler ตอน train บน GPU
        scaler = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))

        best_f1, patience_cnt, best_state = 0.0, 0, None
        max_ep = cfg['max_epochs']

        for epoch in range(max_ep):
            # train 1 epoch
            train_loss = train_epoch(model, train_dl, optimizer, criterion, device, scaler)

            # วัดผลบน validation set
            val_f1 = evaluate_f1(model, val_dl, device, ignore_index=IGNORE_IDX)

            is_best = val_f1 > best_f1
            marker = ' *' if is_best else ''

            print(
                f'    C{combo_idx+1}/{len(combos)} F{fold+1}/5 '
                f'Ep {epoch+1:3d}/{max_ep} '
                f'| Loss {train_loss:.4f} '
                f'| F1 {val_f1:.4f}{marker}',
                end='\r'
            )

            # ถ้ารอบนี้ดีที่สุด ให้เก็บค่าน้ำหนักของโมเดลไว้
            if is_best:
                best_f1, patience_cnt = val_f1, 0
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

            # ถ้าไม่ดีขึ้น ให้นับ patience
            else:
                patience_cnt += 1
                if patience_cnt >= PATIENCE:
                    print(
                        f'    C{combo_idx+1}/{len(combos)} F{fold+1}/5 '
                        f'Ep {epoch+1:3d}/{max_ep} '
                        f'| Loss {train_loss:.4f} '
                        f'| F1 {val_f1:.4f} '
                        f'[Early stop - best F1: {best_f1:.4f}]'
                    )
                    break

        # เซฟโมเดลที่ดีที่สุดของ fold นี้
        if best_state is not None:
            torch.save(best_state, f'best_gnara_c{combo_idx+1:03d}_fold{fold+1}.pt')

        fold_f1s.append(best_f1)

        # ล้างตัวแปรใหญ่หลังจบ fold
        del model, optimizer, train_dl, val_dl, X_train_f, X_val_f
        gc.collect()
        torch.cuda.empty_cache()

    # สรุปผลของ combo นี้จากทุก fold
    mean_f1 = float(np.mean(fold_f1s))
    std_f1 = float(np.std(fold_f1s))

    row = {k: v for k, v in cfg.items() if k not in FIXED_KEYS}
    row.update({
        'mean_f1_weighted': round(mean_f1, 4),
        'std_f1_weighted': round(std_f1, 4)
    })

    all_results.append(row)

    print(f'\n  --> Mean F1: {mean_f1:.4f} +/- {std_f1:.4f}')


  Combo [1/1]
    M: 10
    K: 16
    initial_filters: 32
    max_filters: 256
    learning_rate: 0.001
    focal_alpha: 0.15
    focal_gamma: 2.0


Disabling PyTorch because PyTorch >= 2.4 is required but found 2.2.0
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
/usr/local/lib/python3.10/dist-packages/torch/nn/modules/conv.py:456: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at ../aten/src/ATen/native/Convolution.cpp:1040.)
  return F.conv2d(input, weight, bias, self.stride,


    C1/1 F1/5 Ep  14/50 | Loss 0.0863 | F1 0.3772 [Early stop - best F1: 0.4750]
    C1/1 F2/5 Ep  11/50 | Loss 0.0929 | F1 0.1316 [Early stop - best F1: 0.5229]
    C1/1 F3/5 Ep  14/50 | Loss 0.0935 | F1 0.1095 [Early stop - best F1: 0.5195]
    C1/1 F4/5 Ep  17/50 | Loss 0.0851 | F1 0.0784 [Early stop - best F1: 0.4964]
    C1/1 F5/5 Ep   7/50 | Loss 0.0942 | F1 0.4417 [Early stop - best F1: 0.4644]

  --> Mean F1: 0.4956 +/- 0.0233


In [ ]:
results_df = pd.DataFrame(all_results).sort_values('mean_f1_weighted', ascending=False)
display(results_df)

,M,K,initial_filters,max_filters,learning_rate,focal_alpha,focal_gamma,mean_f1_weighted,std_f1_weighted
0,10,16,32,256,0.001,0.15,2.0,0.4956,0.0233


## 6. Final Training on Full Data
เมื่อได้ Hyperparameters ที่ดีที่สุดแล้ว ให้เทรนด้วยข้อมูลทั้งหมดก่อนนำไปทำนาย Test set

In [ ]:
# เลือก Hyperparameter ที่ดีที่สุดจาก Grid Search (สมมติว่าเป็น Combo 1)
best_cfg = combos[0]
cfg = dict(zip(KEYS, best_cfg))

print(f"🚀 Final Training on ALL data with best config...")
model = GnaraUNet(cfg, in_channels=IN_CHANNELS).to(device)
optimizer = optim.Adam(model.parameters(), lr=cfg['learning_rate'])
criterion = FocalLoss(alpha=cfg['focal_alpha'], gamma=cfg['focal_gamma'], ignore_index=IGNORE_IDX)
scaler = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))

# ใช้ข้อมูลทั้งหมด (X_all, y_all)
full_train_dl = DataLoader(
    TensorDataset(torch.tensor(X_all, dtype=torch.float32), torch.tensor(y_all, dtype=torch.long)),
    batch_size=cfg['batch_size'], shuffle=True, pin_memory=(device.type == 'cuda'), drop_last=False
)

for epoch in range(cfg['max_epochs']):
    loss = train_epoch(model, full_train_dl, optimizer, criterion, device, scaler)
    print(f'Epoch {epoch+1:3d}/{cfg["max_epochs"]} | Loss: {loss:.4f}', end='\r')

torch.save(model.state_dict(), 'final_gnara_model.pt')
print("\n✅ Final model saved as 'final_gnara_model.pt'")

🚀 Final Training on ALL data with best config...
Epoch  50/50 | Loss: 0.0649
✅ Final model saved as 'final_gnara_model.pt'


## 7. Inference & Submission Generation
ประมวลผล Test set และสร้างไฟล์ submission.csv

In [ ]:
TEST_BASE_DIR = r'./data/test_segment/test_segment'  # r'./data/train/train'
SUBMISSION_PATH = r'./data/sample_submission.csv'

# สร้าง map สำหรับแปลงเลขกลับเป็น label เดิม
inv_label_map = {v: k for k, v in LABEL_MAP.items()}

def process_test_subject(subject_folder):
    # ประมวลผลข้อมูลของ 1 subject ใน test set

    subject_id = os.path.basename(subject_folder)

    # หาไฟล์ segment ทั้งหมดแล้วเรียงตามชื่อ
    segment_files = sorted(glob.glob(os.path.join(subject_folder, '*.csv')))

    night_epochs_list = []
    segment_ids = []

    for f in segment_files:
        seg_id = os.path.splitext(os.path.basename(f))[0]
        segment_ids.append(seg_id)

        df = pd.read_csv(f)

        # เตรียม TEMP โดยใช้ค่าการเปลี่ยนแปลงแทนค่าเดิม
        temp = df['TEMP'].values.astype(np.float32)
        temp_diff = np.diff(temp, prepend=temp[0]).astype(np.float32)

        # ทำ preprocessing เหมือนฝั่ง train
        d = {
            'acc_x': np.repeat(df['ACC_X'].values.astype(np.float32), 2),
            'acc_y': np.repeat(df['ACC_Y'].values.astype(np.float32), 2),
            'acc_z': np.repeat(df['ACC_Z'].values.astype(np.float32), 2),
            'bvp':   np.repeat(df['BVP'].values.astype(np.float32), 2),
            'eda':   np.repeat(df['EDA'].values.astype(np.float32), 2),
            'temp_diff': np.repeat(temp_diff, 2)
        }

        d['eda_phasic'] = extract_phasic_eda(d['eda'], FS_TARGET)

        for k in ['acc_x', 'acc_y', 'acc_z', 'bvp', 'eda_phasic']:
            d[k] = adaptive_iqr_normalize(d[k], FS_TARGET, clip_multiplier=20)

        d['temp_diff'] = adaptive_iqr_normalize(d['temp_diff'], FS_TARGET, clip_multiplier=15)

        # แปลงแต่ละสัญญาณเป็น spectrogram
        specs = [
            generate_spectrogram(d[k], FS_TARGET)
            for k in ['acc_x', 'acc_y', 'acc_z', 'bvp', 'eda_phasic', 'temp_diff']
        ]

        seg_specs = np.stack(specs)

        # reshape ให้อยู่ในรูปเดียวกับตอน train
        epoch_feat = seg_specs.transpose(0, 2, 1).reshape(90, 129)
        night_epochs_list.append(epoch_feat.astype(np.float16))

    total_epochs = len(night_epochs_list)

    # รวม segment ทั้งหมดของคืนเดียวเข้าด้วยกันตามแกนเวลา
    night_epochs = np.stack(night_epochs_list, axis=1)

    # padding ให้ยาวเท่ากับที่โมเดลต้องการ
    if total_epochs < MAX_EPOCHS:
        pad_len = MAX_EPOCHS - total_epochs
        night_epochs = np.pad(night_epochs, ((0, 0), (0, pad_len), (0, 0)), mode='constant')
    else:
        night_epochs = night_epochs[:, :MAX_EPOCHS, :]

    # เพิ่ม batch dimension ก่อนส่งเข้าโมเดล
    return torch.tensor(night_epochs, dtype=torch.float32).unsqueeze(0), segment_ids[:MAX_EPOCHS]


# --- Run Inference ---
model.eval()

test_subjects = sorted([
    os.path.join(TEST_BASE_DIR, d)
    for d in os.listdir(TEST_BASE_DIR)
    if os.path.isdir(os.path.join(TEST_BASE_DIR, d))
])

all_preds = {}

with torch.no_grad():
    for subj_path in tqdm(test_subjects, desc='Inference'):
        X_test_night, seg_ids = process_test_subject(subj_path)
        X_test_night = X_test_night.to(device)

        with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
            logits = model(X_test_night)
            preds = logits.argmax(1).cpu().numpy()[0]

        # แปลงผลทำนายกลับเป็น id ของ segment และ label แบบตัวอักษร
        for i, sid in enumerate(seg_ids):
            all_preds[sid] = inv_label_map[preds[i]]

# --- Create Submission ---
sample_sub = pd.read_csv(SUBMISSION_PATH)

# เอาคำตอบที่ทำนายได้ไปใส่ในไฟล์ submission
# ถ้าบาง id ไม่เจอผลทำนาย ให้ใส่ W ไว้ก่อน
sample_sub['labels'] = sample_sub['id'].map(all_preds).fillna('W')

sample_sub.to_csv('submission_gnara.csv', index=False)

print("\n🎉 Submission saved as 'submission_gnara.csv'")
display(sample_sub.head(10))